# LLM Server on Kaggle: Qwen2.5-1.5B-Instruct + vLLM

## 1. Introduction

**vLLM** is an open-source inference and serving engine for large language models. It loads a model onto a GPU and exposes an **OpenAI-compatible HTTP API**, so our local FastAPI backend can call the model in the same way it would call a hosted API.

**Why use a GPU for inference?** A language model performs a huge number of matrix multiplications. A GPU runs thousands of these operations in parallel, so it can generate tokens much faster than a CPU. The model's weights and the key/value (KV) cache also need to live in GPU memory (VRAM).

**What Kaggle provides.** Kaggle Notebooks offer a free NVIDIA GPU (usually two Tesla T4 GPUs, 16 GB each) for a limited session. Kaggle also provides a Linux Python environment with PyTorch and CUDA already installed. We do not need to buy a GPU for this project.

**What this notebook will do.**

1. Confirm that Kaggle gave us a usable CUDA GPU.
2. Check the GPU architecture, Python, PyTorch, and CUDA versions.
3. Install a pinned, compatible version of vLLM.
4. Verify that vLLM imports successfully.

**Loading vs. serving.**

- **Loading** means reading the model's weights from disk into GPU memory and creating a model object. It happens once.
- **Serving** means keeping that loaded model in a long-running process and exposing an API that answers requests. vLLM handles batching, memory management, and token generation for us.

This task stops after installation and verification. The model is **not** downloaded and the server is **not** started yet.

## 2. GPU Verification

Before installing anything, we need to confirm what hardware we actually have.

### CUDA, VRAM, GPU drivers, and compute capability

- **CUDA** is NVIDIA's platform for running general-purpose code on its GPUs. PyTorch and vLLM use CUDA to launch GPU kernels. `torch.version.cuda` shows the CUDA runtime that PyTorch was built with, not the version of the driver.
- **VRAM** (video RAM) is the GPU's own memory. Model weights, activations, and the KV cache must fit inside it. Each T4 has about 16 GB. Running out of VRAM causes an "out of memory" error.
- **A GPU driver** is the low-level software that lets the operating system and CUDA talk to the physical GPU. `nvidia-smi` shows the driver version and the newest CUDA version that driver supports. A new CUDA runtime cannot fix an old driver.
- **Compute capability** identifies the GPU architecture (for example, Turing T4 = 7.5, Pascal P100 = 6.0). vLLM ships compiled CUDA kernels for supported architectures, so the compute capability matters just as much as the amount of VRAM.

The first code cell below stops immediately if CUDA is unavailable. Failing early is better than letting vLLM fail later with a confusing installation error.

In [ ]:
# Fail fast if PyTorch cannot see a CUDA GPU.
# This cell is safe on Kaggle and is not meant to be run on the local Windows machine.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Kaggle, open Settings > Accelerator, "
        "select 'GPU T4 x2', and restart the session before running this notebook."
    )

gpu_index = 0
gpu_name = torch.cuda.get_device_name(gpu_index)
gpu_props = torch.cuda.get_device_properties(gpu_index)

# Report the facts we need for the vLLM compatibility checks.
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU name: {gpu_name}")
print(f"Total VRAM: {gpu_props.total_memory / 1024**3:.2f} GiB")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version (PyTorch build): {torch.version.cuda}")
print(f"Compute capability: {gpu_props.major}.{gpu_props.minor}")

In [ ]:
# nvidia-smi is installed with the NVIDIA driver, not with pip.
# It shows driver information, GPU utilization, VRAM usage, and running processes.
# The "CUDA Version" in its header is the newest CUDA version the driver supports.
!nvidia-smi

## 3. Environment Compatibility

vLLM is not a pure-Python library. Its wheel contains precompiled C++/CUDA kernels, and those kernels are tied to:

- the **Python ABI** (for example, CPython 3.12),
- an exact **PyTorch** release,
- a **CUDA** runtime build,
- the GPU's **compute capability**.

If any one of these differs, the install may appear to work, but importing vLLM or loading a model can fail with errors such as `undefined symbol`, `libcudart.so`, or `no kernel image is available`.

### Why we pin vLLM 0.19.0

| Requirement | vLLM 0.19.0 | Typical Kaggle GPU image |
| --- | --- | --- |
| Operating system | Linux | Linux |
| Python | 3.10 - 3.13 | 3.12 |
| PyTorch | 2.10.0 | 2.10.0 + CUDA 12.8 |
| CUDA | 12.x | 12.8 |
| GPU compute capability | 7.0 or newer | T4 = 7.5 |
| T4 support | Yes | 2 x Tesla T4 |

We choose **vLLM 0.19.0** instead of the newest release because:

- Its published requirements explicitly list compute capability 7.0+ and the Tesla T4.
- It pins PyTorch 2.10.0, which matches the PyTorch version in Kaggle's 2026 T4 image. That means pip should not need to replace Kaggle's PyTorch.
- The newest vLLM release may be built for a newer PyTorch/CUDA/driver stack. It might still support the T4, but installing it could force a large, unnecessary PyTorch upgrade.

**Pascal P100 warning:** Kaggle also offers a P100 accelerator. The P100 has compute capability 6.0, which is below vLLM 0.19.0's minimum of 7.0. If the next cells show a P100, switch the Kaggle accelerator to **GPU T4 x2** and restart. Do not try to force vLLM onto unsupported hardware.

The next cell only reports the environment. The installation cell later turns these requirements into hard checks.

In [ ]:
# Print the environment facts that affect vLLM compatibility.
import platform
import sys

import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"Operating system: {platform.platform()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA runtime (PyTorch build): {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Visible GPU count: {torch.cuda.device_count()}")
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(
            f"  GPU {index}: {props.name} | "
            f"{props.total_memory / 1024**3:.2f} GiB | "
            f"compute capability {props.major}.{props.minor}"
        )
else:
    print("Visible GPU: none")

## 4. Install vLLM

The cell below installs exactly **vLLM 0.19.0**. It runs only on Kaggle.

What the cell does **before** installing:

1. Refuses to run on Windows or macOS (vLLM does not support them natively).
2. Checks Python 3.10-3.13.
3. Checks that PyTorch is exactly 2.10.0 (ignoring the `+cu128` local suffix).
4. Checks that PyTorch can see CUDA and uses CUDA 12.x.
5. Checks that the GPU's compute capability is at least 7.0 (T4 = 7.5).

If a check fails, the cell raises an error and **stops before pip runs**. That is intentional: a clear compatibility error is easier to fix than a broken installation.

The `%pip install` command:

- Uses the same Python environment as this notebook kernel. This is safer in a notebook than calling `!pip`.
- Pins the version, so future runs are reproducible.
- Does **not** use `--no-deps`, because vLLM needs its matching dependencies.
- Does **not** force or upgrade PyTorch itself. vLLM 0.19.0 declares `torch==2.10.0`, and Kaggle already provides that version.

The download is large (roughly several GB including CUDA libraries and `flashinfer`), so it can take 5-15 minutes. Do not interrupt it.

In [ ]:
# =============================================================================
# KAGGLE ONLY: install the pinned vLLM build.
# Run this cell only after the GPU checks above pass.
# Do not run this cell in the local Windows environment.
# =============================================================================
import sys

import torch

VLLM_VERSION = "0.19.0"
EXPECTED_TORCH_VERSION = "2.10.0"
MIN_COMPUTE_CAPABILITY = (7, 0)

# vLLM publishes Linux wheels only.
if not sys.platform.startswith("linux"):
    raise RuntimeError(
        "vLLM does not support Windows natively. Run this notebook on Kaggle "
        "(Linux) instead of the local Windows machine."
    )

# vLLM 0.19.0 supports Python 3.10 through 3.13.
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} needs Python 3.10-3.13, but this kernel is "
        f"Python {sys.version.split()[0]}. Do not force the install; use a "
        "Kaggle image with a supported Python version."
    )

# vLLM 0.19.0 is built against PyTorch 2.10.0. Stop instead of letting pip
# replace a different Kaggle PyTorch build.
installed_torch = torch.__version__.split("+")[0]
if installed_torch != EXPECTED_TORCH_VERSION:
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} expects PyTorch {EXPECTED_TORCH_VERSION}, but "
        f"this kernel has PyTorch {torch.__version__}. The Kaggle image may "
        "have changed. Do not force-install. Record the environment output "
        "above and choose a vLLM version that pins this PyTorch release."
    )

# vLLM needs a visible CUDA 12.x build of PyTorch.
if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot see a CUDA GPU. In Kaggle, enable Settings > "
        "Accelerator > GPU T4 x2 and restart the session."
    )
if torch.version.cuda is None or not torch.version.cuda.startswith("12."):
    raise RuntimeError(
        f"vLLM {VLLM_VERSION} requires a CUDA 12.x PyTorch build, but PyTorch "
        f"reports CUDA {torch.version.cuda}. Do not force the install."
    )

# A T4 is compute capability 7.5. A P100 is 6.0 and is NOT supported.
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
if (major, minor) < MIN_COMPUTE_CAPABILITY:
    raise RuntimeError(
        f"GPU '{gpu_name}' has compute capability {major}.{minor}, but vLLM "
        f"{VLLM_VERSION} requires 7.0 or newer. The Tesla P100 (6.0) is not "
        "supported. In Kaggle, switch to 'GPU T4 x2' and restart the session. "
        "Do not attempt random dependency fixes."
    )

print("Compatibility checks passed.")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.version.cuda}")
print(f"  GPU: {gpu_name} (compute capability {major}.{minor})")
print(f"\nInstalling vLLM {VLLM_VERSION}. This can take 5-15 minutes...")

# %pip installs into the kernel's Python environment. --no-cache-dir avoids
# filling Kaggle's limited disk with a second copy of the downloaded wheels.
# We deliberately do NOT use --no-deps: vLLM's required dependencies must be
# resolved normally so the compiled kernels and Python packages stay consistent.
%pip install "vllm==0.19.0" --no-cache-dir --progress-bar off

### Restart the kernel after installation

`%pip` writes the new packages to disk, but the current Python process has already imported PyTorch and loaded native libraries. It may continue using old modules, or it may fail to load vLLM's newly installed CUDA extensions.

After the install finishes successfully:

1. Restart **the notebook kernel**. In Kaggle use the notebook's restart option; in Jupyter or VS Code use **Kernel > Restart**. Do not use "Stop Session" or a factory reset, because that can discard the packages you just installed.
2. Do **not** rerun the install cell.
3. Run the verification cell in Section 5.

If your environment only offers a full reset that discards installed packages, rerun the install cell once after it comes back, then run verification. The important point is that a fresh Python process must import the newly installed packages.

## 5. Verify the Installation

The verification cell:

- imports `vllm`,
- checks that the installed version is exactly 0.19.0,
- checks that PyTorch is still 2.10.0,
- checks that CUDA is still visible,
- prints a success message only when all checks pass.

If the import fails, the error message lists the usual causes. vLLM may print non-fatal warnings about unavailable optimizations (for example, FlashAttention on a T4). A warning is not a failure; a raised exception is.

In [ ]:
# Run this cell AFTER restarting the kernel. Do not rerun the install cell.
import torch

EXPECTED_VLLM_VERSION = "0.19.0"
EXPECTED_TORCH_VERSION = "2.10.0"
MIN_COMPUTE_CAPABILITY = (7, 0)

try:
    import vllm
except Exception as exc:
    raise RuntimeError(
        "vLLM could not be imported. Check the following:\n"
        "1. Did the install cell finish without errors?\n"
        "2. Did you restart the kernel after installing?\n"
        "3. Is a GPU still enabled in Kaggle?\n"
        "4. Does the error mention 'undefined symbol', 'libcudart.so', or "
        "'libcuda.so'? That usually means a stale kernel or a PyTorch/CUDA "
        "binary mismatch.\n"
        "Do not force-install other versions. Copy the full error log."
    ) from exc

installed_version = getattr(vllm, "__version__", "unknown")
if installed_version != EXPECTED_VLLM_VERSION:
    raise RuntimeError(
        f"Expected vLLM {EXPECTED_VLLM_VERSION}, but found {installed_version}. "
        "Rerun the pinned install cell and restart the kernel."
    )

if torch.__version__.split("+")[0] != EXPECTED_TORCH_VERSION:
    raise RuntimeError(
        f"PyTorch changed after installing vLLM: {torch.__version__}. Expected "
        f"{EXPECTED_TORCH_VERSION}. Do not force the install; share this output."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available after installing vLLM. Enable the Kaggle GPU "
        "accelerator, restart the kernel, and run this cell again."
    )

major, minor = torch.cuda.get_device_capability(0)
if (major, minor) < MIN_COMPUTE_CAPABILITY:
    raise RuntimeError(
        f"GPU compute capability {major}.{minor} is unsupported by vLLM "
        f"{EXPECTED_VLLM_VERSION}; 7.0 or newer is required."
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"vLLM version: {installed_version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {gpu_name} (compute capability {major}.{minor})")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\nSUCCESS: vLLM is installed and CUDA is accessible.")
print("The environment is ready for model loading in the next task.")

## 6. What Just Happened?

### During installation

- pip downloaded the vLLM 0.19.0 wheel (compiled CUDA kernels), its Python dependencies, `flashinfer`, and CUDA-linked libraries.
- Because vLLM 0.19.0 pins `torch==2.10.0`, pip reused Kaggle's existing PyTorch instead of replacing it. The verification cell confirms this.
- No model weights were downloaded. Installing vLLM is like installing an engine; the model itself is not included.
- The kernel restart unloaded the old Python modules so the next process could import the new native libraries cleanly.

### Why vLLM requires compatible GPU hardware

- vLLM's wheel contains machine code compiled for specific GPU architectures. A T4 is Turing (compute capability 7.5). vLLM 0.19.0 supports 7.0 and newer, but not Pascal (P100, 6.0).
- CUDA kernels also depend on the CUDA runtime and PyTorch ABI they were built with. Mixing a vLLM wheel with an unrelated PyTorch or CUDA build can produce import errors or crashes, even when the Python package appears to install.
- Newer vLLM releases may target newer CUDA/driver stacks. Pinning a version that matches Kaggle's image is safer and reproducible.

### What model inference means

- The model turns the input text into **tokens** (small pieces of words).
- Each token becomes a vector, and the transformer performs many matrix multiplications to predict the most likely next token.
- The generated token is appended to the input, and the process repeats. This is **autoregressive generation**.
- A GPU speeds up those matrix operations. VRAM holds the model weights, temporary activations, and the KV cache. The KV cache remembers earlier tokens so the model does not recompute the whole prompt for every new token.
- **Serving** keeps the model loaded and answers many requests through an API. That is why we install an inference server instead of reloading the model for every request.

### Why Qwen2.5-1.5B-Instruct?

- **Small enough for a T4.** It has about 1.5 billion parameters; in fp16 the weights are roughly 3 GB, leaving plenty of the T4's 16 GB for the KV cache and activations.
- **Instruction-tuned.** It follows user instructions and is suitable for chat-style RAG answers.
- **Good language coverage.** Qwen2.5 supports many languages and up to 32K tokens of context, which helps when we pass retrieved document chunks.
- **Permissive license.** The Qwen2.5-1.5B-Instruct model is released under the Apache 2.0 license, which is convenient for learning and experimentation.
- **Practical for free hardware.** A 1.5B model starts faster and generates tokens faster on a T4 than a 7B or larger model.

### What the next development step will accomplish

The next task will load `Qwen/Qwen2.5-1.5B-Instruct` with vLLM and start its OpenAI-compatible server on port 8000. We will then send a small test request to confirm that the server can generate text. Tunneling, the local FastAPI backend, LangChain, and the RAG pipeline still come later.

For now, stop here: the GPU works, vLLM is installed, and the environment is verified. Do not start the server or download the model yet.